In [43]:
from contextlib import ExitStack
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

from IPython.display import display
from tqdm.auto import tqdm

# VNP46A1 acquisition-condition and observability diagnostics

This notebook diagnoses whether VNP46A1 acquisition conditions could confound the interpretation of VNP46A2 nighttime lights around Typhoon Haiyan. It:

- Automatically discovers and combines all six GeoTIFF stacks.
- Resolves overlapping dates consistently.
- Detects whether the exported values are already scaled.
- Prevents the double-scaling present in the previous notebook.
- Reads all eight bands for each date in one raster operation.
- Saves checkpoints for each GeoTIFF so interrupted runs can resume.
- Reuses cached results in later sessions.
- Accumulates spatial diagnostics during the daily loop, avoiding a second full raster pass.
- Uses robust Plotly axis ranges and NumPy arrays to prevent the previous plotting errors.

VNP46A1 contains at-sensor radiance and acquisition conditions. It is used here diagnostically; VNP46A2 DNB-BRDF remains the primary recovery signal.


## 1. Imports

This cell loads the raster-processing, tabular, progress-reporting, and Plotly dependencies used throughout the notebook.


In [44]:
from contextlib import ExitStack
from pathlib import Path
import hashlib
import json
import re
import warnings

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display, Markdown
from tqdm.auto import tqdm

## 2. Paths and processing controls

The filename pattern automatically includes all six VNP46A1 stacks. Each stack receives an independent cache, allowing completed stacks to be skipped during later runs.

Leave `FORCE_REPROCESS = False` for normal use. Change it to `True` only when the rasters, scaling, denominator, or screening logic changes.


In [45]:
DATA_DIR = Path("../datasets/vnp46a1")
INPUT_PATTERN = "Haiyan_VNP46A1_KeyBands_*"

INPUT_FILES = sorted(
    path
    for path in DATA_DIR.glob(INPUT_PATTERN)
    if path.suffix.lower() in {".tif", ".tiff"}
)

OUTPUT_DIR = Path(
    "outputs/haiyan_vnp46a1_diagnostics"
)

OUTPUT_TABLE_DIR = OUTPUT_DIR / "tables"
OUTPUT_FIGURE_DIR = OUTPUT_DIR / "figures"
CACHE_DIR = OUTPUT_DIR / "cache"

for directory in [
    OUTPUT_TABLE_DIR,
    OUTPUT_FIGURE_DIR,
    CACHE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

if not INPUT_FILES:
    raise FileNotFoundError(
        f"No VNP46A1 GeoTIFFs match "
        f"{DATA_DIR / INPUT_PATTERN}."
    )


EVENT_DATE = pd.Timestamp("2013-11-08")
NO_DATA = -9999.0

# "auto" distinguishes Earth Engine physical values
# from raw product integers.
SCALING_MODE = "auto"

OBSERVABILITY_DENOMINATOR = (
    "land_and_coast"
)

SUPPORT_SAMPLE_DATES = 24
PROFILE_EVERY_N_DAYS = 30
CHECKPOINT_EVERY_N_DAYS = 25
MAX_MAP_DIMENSION = 700

PROCESSING_VERSION = (
    "v3_batch_read_scaling_fix"
)

FORCE_REPROCESS = False
SAVE_OUTPUTS = True
SAVE_PNG = False

## 3. VNP46A1 band metadata

The Earth Engine VNP46A1 catalogue reports physical band ranges, whereas the NASA product guide also documents scale factors for raw HDF integers. The notebook therefore diagnoses the stored values before applying any scale factor.

This prevents Earth Engine radiance, VZA, azimuth, and moon illumination from being scaled twice.

Sources: [Earth Engine VNP46A1 catalogue](https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_001_VNP46A1) and [NASA Black Marble User Guide](https://viirsland.gsfc.nasa.gov/PDF/BlackMarbleUserGuide_v1.2_20220916.pdf).


In [46]:
EXPECTED_BANDS = [
    "DNB_At_Sensor_Radiance_500m",
    "Sensor_Zenith",
    "Sensor_Azimuth",
    "UTC_Time",
    "Granule",
    "QF_DNB",
    "QF_Cloud_Mask",
    "Moon_Illumination_Fraction",
]


BAND_META = {
    "DNB_At_Sensor_Radiance_500m": {
        "product_scale": 0.1,
        "valid_range": (0, 6553.4),
        "fill_values": [65535],
        "units": "nW cm^-2 sr^-1",
    },
    "Sensor_Zenith": {
        "product_scale": 0.01,
        "valid_range": (0, 90),
        "fill_values": [-32768],
        "units": "degrees",
    },
    "Sensor_Azimuth": {
        "product_scale": 0.01,
        "valid_range": (-180, 180),
        "fill_values": [-32768],
        "units": "degrees",
    },
    "UTC_Time": {
        "product_scale": 1.0,
        "valid_range": (0, 24),
        "fill_values": [-999.9],
        "units": "decimal UTC hours",
    },
    "Granule": {
        "product_scale": 1.0,
        "valid_range": (0, 254),
        "fill_values": [255],
        "units": "granule number",
    },
    "QF_DNB": {
        "product_scale": 1.0,
        "valid_range": (0, 65534),
        "fill_values": [65535],
        "units": "bit flags",
    },
    "QF_Cloud_Mask": {
        "product_scale": 1.0,
        "valid_range": (0, 65534),
        "fill_values": [65535],
        "units": "bit flags",
    },
    "Moon_Illumination_Fraction": {
        "product_scale": 0.01,
        "valid_range": (0, 100),
        "fill_values": [-32768],
        "units": "percent",
    },
}


QF_DNB_FLAGS = {
    "Substitute_calibration": 1,
    "Out_of_range": 2,
    "Saturation": 4,
    "Temperature_not_nominal": 8,
    "Stray_light": 16,
    "Bowtie_deleted": 256,
    "Missing_EV": 512,
    "Calibration_failure": 1024,
    "Dead_detector": 2048,
}


LAND_WATER_VALUES = np.array(
    [0, 1, 2, 3, 5],
    dtype=np.int16,
)

LAND_WATER_LABELS = {
    0: "Land and desert",
    1: "Land, no desert",
    2: "Inland water",
    3: "Sea water",
    5: "Coastal",
}

## 4. Band-description and quality-mask helpers

These functions parse acquisition dates and variable names from the GeoTIFF band descriptions, create an inventory for every stack, and decode the VNP46A1 cloud-mask bits.


In [47]:
def parse_band_description(
    description,
    band_index,
):
    description = (
        ""
        if description is None
        else str(description)
    )

    date_match = re.search(
        r"((?:19|20)\d{2})[_-]?(\d{2})[_-]?(\d{2})",
        description,
    )

    date = pd.NaT

    if date_match is not None:
        date = pd.Timestamp(
            year=int(date_match.group(1)),
            month=int(date_match.group(2)),
            day=int(date_match.group(3)),
        )

    variable = next(
        (
            band
            for band in sorted(
                EXPECTED_BANDS,
                key=len,
                reverse=True,
            )
            if band in description
        ),
        None,
    )

    return {
        "band_index": band_index,
        "description": description,
        "date": date,
        "variable": variable,
    }


def create_band_inventory(input_files):
    metadata_rows = []
    file_rows = []

    for file_priority, file_path in enumerate(
        input_files
    ):
        with rasterio.open(file_path) as src:
            file_rows.append({
                "file": str(file_path),
                "file_priority": file_priority,
                "width": src.width,
                "height": src.height,
                "band_count": src.count,
                "dtype": src.dtypes[0],
                "nodata": src.nodata,
                "crs": (
                    src.crs.to_string()
                    if src.crs is not None
                    else None
                ),
                "transform": tuple(
                    src.transform
                )[:6],
                "bounds": tuple(src.bounds),
                "size_gb": (
                    file_path.stat().st_size
                    / 1024**3
                ),
            })

            for band_index, description in enumerate(
                src.descriptions,
                start=1,
            ):
                if description is None:
                    band_tags = src.tags(
                        band_index
                    )

                    description = (
                        band_tags.get(
                            "DESCRIPTION"
                        )
                        or band_tags.get(
                            "description"
                        )
                        or band_tags.get("NAME")
                    )

                parsed = parse_band_description(
                    description,
                    band_index,
                )

                parsed.update({
                    "file": str(file_path),
                    "file_priority": (
                        file_priority
                    ),
                })

                metadata_rows.append(parsed)

    return (
        pd.DataFrame(metadata_rows),
        pd.DataFrame(file_rows),
    )


def to_uint16(values):
    integers = np.zeros(
        values.shape,
        dtype=np.uint16,
    )

    valid = np.isfinite(values)

    integers[valid] = (
        np.rint(values[valid])
        .astype(np.uint16)
    )

    return integers, valid


def decode_cloud_mask(values):
    integers, valid = to_uint16(values)

    return {
        "valid": valid,
        "day_night": integers & 1,
        "land_water": (
            integers >> 1
        ) & 7,
        "mask_quality": (
            integers >> 4
        ) & 3,
        "cloud_confidence": (
            integers >> 6
        ) & 3,
        "shadow": (
            integers >> 8
        ) & 1,
        "cirrus": (
            integers >> 9
        ) & 1,
        "snow_ice": (
            integers >> 10
        ) & 1,
    }

## 5. Inventory the six GeoTIFF stacks

This cell validates that all stacks use the same spatial grid, identifies complete eight-band dates, and resolves overlapping dates by retaining the later file in `INPUT_FILES`.

The output table should show the temporal coverage contributed by each stack. Missing dates remain explicit rather than being silently dropped.


In [48]:
band_metadata, file_inventory = (
    create_band_inventory(INPUT_FILES)
)


unparsed_bands = band_metadata[
    band_metadata["date"].isna()
    | band_metadata["variable"].isna()
]

if not unparsed_bands.empty:
    display(
        unparsed_bands[
            [
                "file",
                "band_index",
                "description",
            ]
        ].head(30)
    )

    raise ValueError(
        "Some GeoTIFF bands could not be parsed. "
        "Inspect their descriptions."
    )


grid_signatures = {
    (
        row.width,
        row.height,
        row.crs,
        tuple(row.transform),
        tuple(row.bounds),
    )
    for row in file_inventory.itertuples(
        index=False
    )
}

if len(grid_signatures) != 1:
    raise ValueError(
        "All stacks must share one CRS, extent, "
        "transform, width, and height."
    )


valid_metadata = band_metadata[
    band_metadata["variable"].isin(
        EXPECTED_BANDS
    )
].copy()


date_file_counts = (
    valid_metadata
    .groupby(
        [
            "date",
            "file",
            "file_priority",
        ],
        as_index=False,
    )
    .agg(
        Available_bands=(
            "variable",
            "nunique",
        )
    )
)


complete_candidates = date_file_counts[
    date_file_counts["Available_bands"]
    == len(EXPECTED_BANDS)
].copy()

if complete_candidates.empty:
    raise ValueError(
        "No date-file combination contains "
        "all eight bands."
    )


overlap_counts = (
    complete_candidates
    .groupby("date")
    .size()
)

overlap_dates = overlap_counts[
    overlap_counts > 1
].index

if len(overlap_dates) > 0:
    warnings.warn(
        f"{len(overlap_dates)} dates occur in "
        "overlapping complete stacks. The later "
        "file in INPUT_FILES is used."
    )


selected_sources = (
    complete_candidates
    .sort_values(
        [
            "date",
            "file_priority",
        ]
    )
    .drop_duplicates(
        "date",
        keep="last",
    )
    [
        [
            "date",
            "file",
            "file_priority",
        ]
    ]
)


selected_metadata = (
    valid_metadata
    .merge(
        selected_sources,
        on=[
            "date",
            "file",
            "file_priority",
        ],
        how="inner",
    )
)


band_lookup = (
    selected_metadata
    .set_index(
        [
            "date",
            "variable",
        ]
    )
    [
        [
            "file",
            "band_index",
        ]
    ]
    .to_dict("index")
)


complete_dates = sorted(
    pd.to_datetime(
        selected_sources["date"]
    ).tolist()
)

expected_dates = pd.date_range(
    min(complete_dates),
    max(complete_dates),
    freq="D",
)

selected_source_map = (
    selected_sources
    .set_index("date")["file"]
    .to_dict()
)


date_inventory_rows = []

for date in expected_dates:
    source_file = selected_source_map.get(
        pd.Timestamp(date)
    )

    if source_file is None:
        available = set(
            valid_metadata.loc[
                valid_metadata["date"] == date,
                "variable",
            ]
        )
    else:
        available = set(
            selected_metadata.loc[
                (
                    selected_metadata["date"]
                    == date
                )
                & (
                    selected_metadata["file"]
                    == source_file
                ),
                "variable",
            ]
        )

    missing = [
        band
        for band in EXPECTED_BANDS
        if band not in available
    ]

    date_inventory_rows.append({
        "Date": date,
        "Source_file": (
            Path(source_file).name
            if source_file
            else None
        ),
        "Available_bands": len(available),
        "Missing_bands": ", ".join(
            missing
        ),
        "Complete_band_set": (
            source_file is not None
            and not missing
        ),
    })


date_inventory_df = pd.DataFrame(
    date_inventory_rows
)


file_date_summary = (
    selected_sources
    .groupby(
        "file",
        as_index=False,
    )
    .agg(
        Start_date=(
            "date",
            "min",
        ),
        End_date=(
            "date",
            "max",
        ),
        Selected_dates=(
            "date",
            "nunique",
        ),
    )
    .merge(
        file_inventory[
            [
                "file",
                "band_count",
                "size_gb",
            ]
        ],
        on="file",
        how="left",
    )
)

file_date_summary["file"] = (
    file_date_summary["file"].map(
        lambda value: Path(value).name
    )
)


display(
    file_date_summary.round({
        "size_gb": 3
    })
)

display(
    date_inventory_df.loc[
        ~date_inventory_df[
            "Complete_band_set"
        ]
    ].head(30)
)


print(
    "GeoTIFF stacks discovered:",
    len(INPUT_FILES),
)

print(
    "Expected calendar days:",
    len(expected_dates),
)

print(
    "Complete dates selected:",
    len(complete_dates),
)

print(
    "Selected GeoTIFF bands:",
    len(selected_metadata),
)

,file,Start_date,End_date,Selected_dates,band_count,size_gb
0,Haiyan_VNP46A1_KeyBands_PostEvent_D0_D180.tif,2013-11-08,2014-05-07,181,1448,0.398
1,Haiyan_VNP46A1_KeyBands_PreEvent_D180_D1.tif,2013-05-12,2013-11-07,180,1440,0.388


,Date,Source_file,Available_bands,Missing_bands,Complete_band_set


GeoTIFF stacks discovered: 2
Expected calendar days: 361
Complete dates selected: 361
Selected GeoTIFF bands: 2888


## 6. Batch-reading and scaling diagnosis

`read_date_arrays()` reads all requested bands from a stack in one operation. This replaces eight separate raster reads per date.

The scaling diagnostic samples multiple dates. Values are treated as raw product integers only when the angle or moon-illumination ranges exceed their physical limits.

For the Earth Engine exports used here, the expected result is `Scaling scheme selected: earth_engine`.


In [49]:
def prepare_band(
    values,
    variable,
    source_nodata,
    scale_factor=1.0,
):
    values = values.astype(
        np.float32,
        copy=False,
    )

    invalid = (
        ~np.isfinite(values)
        | np.isclose(
            values,
            NO_DATA,
        )
    )

    if source_nodata is not None:
        invalid |= np.isclose(
            values,
            source_nodata,
        )

    for fill_value in BAND_META[
        variable
    ]["fill_values"]:
        invalid |= np.isclose(
            values,
            fill_value,
        )

    values = values.copy()
    values[invalid] = np.nan

    if scale_factor != 1.0:
        values *= np.float32(
            scale_factor
        )

    return values


def read_date_arrays(
    sources,
    lookup,
    date,
    variables=None,
    scale_factors=None,
    out_shape=None,
):
    date = pd.Timestamp(date)

    variables = (
        EXPECTED_BANDS
        if variables is None
        else list(variables)
    )

    records_by_file = {}

    for variable in variables:
        record = lookup.get(
            (
                date,
                variable,
            )
        )

        if record is None:
            raise KeyError(
                f"Missing {variable} for "
                f"{date.date()}."
            )

        records_by_file.setdefault(
            record["file"],
            [],
        ).append(
            (
                variable,
                int(record["band_index"]),
            )
        )

    arrays = {}

    for file_name, records in (
        records_by_file.items()
    ):
        src = sources[file_name]

        indexes = [
            band_index
            for _, band_index in records
        ]

        read_kwargs = {}

        if out_shape is not None:
            read_kwargs.update({
                "out_shape": (
                    len(indexes),
                    *out_shape,
                ),
                "resampling": (
                    Resampling.nearest
                ),
            })

        data = src.read(
            indexes,
            **read_kwargs,
        )

        for layer, (
            variable,
            _,
        ) in zip(
            data,
            records,
        ):
            scale_factor = (
                1.0
                if scale_factors is None
                else scale_factors[
                    variable
                ]
            )

            arrays[variable] = (
                prepare_band(
                    layer,
                    variable,
                    src.nodata,
                    scale_factor,
                )
            )

    return arrays


def finite_percentile(
    values,
    percentile,
):
    finite = values[
        np.isfinite(values)
    ]

    if finite.size == 0:
        return np.nan

    return float(
        np.percentile(
            finite,
            percentile,
        )
    )


scaling_sample_dates = [
    complete_dates[index]
    for index in np.unique(
        np.linspace(
            0,
            len(complete_dates) - 1,
            min(
                3,
                len(complete_dates),
            ),
        )
        .round()
        .astype(int)
    )
]


scaling_rows = []

with ExitStack() as stack:
    sources = {
        str(path): stack.enter_context(
            rasterio.open(path)
        )
        for path in INPUT_FILES
    }

    for date in scaling_sample_dates:
        sample = read_date_arrays(
            sources,
            band_lookup,
            date,
            variables=[
                "DNB_At_Sensor_Radiance_500m",
                "Sensor_Zenith",
                "Sensor_Azimuth",
                "Moon_Illumination_Fraction",
            ],
            scale_factors=None,
        )

        for variable, values in (
            sample.items()
        ):
            scaling_rows.append({
                "Date": date,
                "Variable": variable,
                "Raw_p99": (
                    finite_percentile(
                        np.abs(values),
                        99,
                    )
                ),
                "Raw_max": (
                    finite_percentile(
                        np.abs(values),
                        100,
                    )
                ),
            })


scaling_evidence_df = (
    pd.DataFrame(scaling_rows)
)

scaling_evidence_summary = (
    scaling_evidence_df
    .groupby(
        "Variable",
        as_index=False,
    )
    .agg(
        Raw_p99=(
            "Raw_p99",
            "median",
        ),
        Raw_max=(
            "Raw_max",
            "max",
        ),
    )
)


if SCALING_MODE not in {
    "auto",
    "earth_engine",
    "raw_product",
}:
    raise ValueError(
        "SCALING_MODE must be 'auto', "
        "'earth_engine', or 'raw_product'."
    )


if SCALING_MODE == "auto":
    evidence = (
        scaling_evidence_summary
        .set_index("Variable")
    )

    raw_votes = []

    for variable, threshold in {
        "Sensor_Zenith": 100,
        "Sensor_Azimuth": 200,
        "Moon_Illumination_Fraction": 101,
    }.items():
        value = evidence.loc[
            variable,
            "Raw_p99",
        ]

        if np.isfinite(value):
            raw_votes.append(
                value > threshold
            )

    if not raw_votes:
        warnings.warn(
            "Scaling could not be inferred "
            "from the geometry bands. "
            "Earth Engine physical units "
            "are assumed."
        )

        SCALING_SCHEME = (
            "earth_engine"
        )
    else:
        SCALING_SCHEME = (
            "raw_product"
            if any(raw_votes)
            else "earth_engine"
        )
else:
    SCALING_SCHEME = SCALING_MODE


APPLIED_SCALE_FACTORS = {
    variable: (
        metadata["product_scale"]
        if SCALING_SCHEME
        == "raw_product"
        else 1.0
    )
    for variable, metadata
    in BAND_META.items()
}


scaling_diagnostic_df = (
    scaling_evidence_summary.copy()
)

scaling_diagnostic_df[
    "Applied_scale"
] = scaling_diagnostic_df[
    "Variable"
].map(
    APPLIED_SCALE_FACTORS
)

scaling_diagnostic_df[
    "Scaled_p99"
] = (
    scaling_diagnostic_df["Raw_p99"]
    * scaling_diagnostic_df[
        "Applied_scale"
    ]
)

scaling_diagnostic_df[
    "Physical_range"
] = scaling_diagnostic_df[
    "Variable"
].map(
    lambda variable: str(
        BAND_META[
            variable
        ]["valid_range"]
    )
)


display(
    scaling_diagnostic_df.round(4)
)

print(
    "Scaling scheme selected:",
    SCALING_SCHEME,
)

,Variable,Raw_p99,Raw_max,Applied_scale,Scaled_p99,Physical_range
0,DNB_At_Sensor_Radiance_500m,1.00,95.90,1.0,1.00,"(0, 6553.4)"
1,Moon_Illumination_Fraction,32.57,55.39,1.0,32.57,"(0, 100)"
2,Sensor_Azimuth,97.23,179.92,1.0,97.23,"(-180, 180)"
3,Sensor_Zenith,50.65,66.24,1.0,50.65,"(0, 90)"


Scaling scheme selected: earth_engine


## 7. Construct a stable spatial denominator

The previous notebook used the land/water class from one date. The observed daily classification changed by up to approximately 4.6%, making that denominator sensitive to the selected reference date.

This cell estimates the modal land/water class across evenly distributed dates. The resulting mask remains fixed throughout the analysis.


In [50]:
def evenly_spaced_dates(
    dates,
    number,
):
    indexes = np.unique(
        np.linspace(
            0,
            len(dates) - 1,
            min(
                number,
                len(dates),
            ),
        )
        .round()
        .astype(int)
    )

    return [
        dates[index]
        for index in indexes
    ]


support_dates = evenly_spaced_dates(
    complete_dates,
    SUPPORT_SAMPLE_DATES,
)


reference_path = Path(
    selected_source_map[
        complete_dates[0]
    ]
)

with rasterio.open(
    reference_path
) as reference_src:
    native_height = (
        reference_src.height
    )

    native_width = (
        reference_src.width
    )

    native_shape = (
        native_height,
        native_width,
    )

    native_transform = (
        reference_src.transform
    )

    native_crs = (
        reference_src.crs
    )


valid_support_count = np.zeros(
    native_shape,
    dtype=np.uint16,
)

land_water_counts = np.zeros(
    (
        len(LAND_WATER_VALUES),
        *native_shape,
    ),
    dtype=np.uint16,
)


with ExitStack() as stack:
    sources = {
        str(path): stack.enter_context(
            rasterio.open(path)
        )
        for path in INPUT_FILES
    }

    for date in tqdm(
        support_dates,
        desc=(
            "Building stable "
            "land/water support"
        ),
        unit="date",
    ):
        qf_cloud = read_date_arrays(
            sources,
            band_lookup,
            date,
            variables=[
                "QF_Cloud_Mask"
            ],
            scale_factors=(
                APPLIED_SCALE_FACTORS
            ),
        )["QF_Cloud_Mask"]

        cloud = decode_cloud_mask(
            qf_cloud
        )

        valid_support_count += (
            cloud["valid"]
            .astype(np.uint16)
        )

        for position, class_value in (
            enumerate(
                LAND_WATER_VALUES
            )
        ):
            land_water_counts[
                position
            ] += (
                cloud["valid"]
                & (
                    cloud[
                        "land_water"
                    ] == class_value
                )
            ).astype(np.uint16)


minimum_support_votes = max(
    1,
    int(
        np.ceil(
            len(support_dates)
            / 2
        )
    ),
)

reference_roi_mask = (
    valid_support_count
    >= minimum_support_votes
)


modal_positions = np.argmax(
    land_water_counts,
    axis=0,
)

reference_land_water = (
    LAND_WATER_VALUES[
        modal_positions
    ]
)

reference_land_water[
    land_water_counts.max(
        axis=0
    ) == 0
] = -1


reference_land_mask = (
    reference_roi_mask
    & np.isin(
        reference_land_water,
        [0, 1, 5],
    )
)


if (
    OBSERVABILITY_DENOMINATOR
    == "land_and_coast"
):
    support_mask = (
        reference_land_mask
    )

elif (
    OBSERVABILITY_DENOMINATOR
    == "all_roi"
):
    support_mask = (
        reference_roi_mask
    )

else:
    raise ValueError(
        "OBSERVABILITY_DENOMINATOR "
        "must be 'land_and_coast' "
        "or 'all_roi'."
    )


support_pixel_count = int(
    np.count_nonzero(
        support_mask
    )
)

if support_pixel_count == 0:
    raise ValueError(
        "The stable observability "
        "denominator contains no pixels."
    )


support_summary_df = pd.DataFrame([
    {
        "Support_dates": (
            len(support_dates)
        ),
        "Minimum_votes": (
            minimum_support_votes
        ),
        "Raster_pixels": int(
            np.prod(native_shape)
        ),
        "Stable_ROI_pixels": int(
            np.count_nonzero(
                reference_roi_mask
            )
        ),
        "Land_and_coast_pixels": int(
            np.count_nonzero(
                reference_land_mask
            )
        ),
        "Analysis_support_pixels": (
            support_pixel_count
        ),
        "Denominator": (
            OBSERVABILITY_DENOMINATOR
        ),
    }
])


display(support_summary_df)

Building stable land/water support: 100%|██████████| 24/24 [01:09<00:00,  2.90s/date]


,Support_dates,Minimum_votes,Raster_pixels,Stable_ROI_pixels,Land_and_coast_pixels,Analysis_support_pixels,Denominator
0,24,12,318802,103801,102760,102760,land_and_coast


## 8. Daily-processing and checkpoint helpers

These functions calculate the daily observability, geometry, radiance, and quality diagnostics.

Physical-value profiles are calculated only every 30 days, plus the first, last, and nearest-Haiyan dates. This is sufficient for scaling diagnostics without repeatedly calculating expensive full-array percentiles.

Each stack is checkpointed every 25 processed dates.


In [ ]:
def percentage(
    mask,
    denominator_mask=support_mask,
):
    denominator = np.count_nonzero(
        denominator_mask
    )

    if denominator == 0:
        return np.nan

    return (
        np.count_nonzero(
            mask & denominator_mask
        )
        / denominator
        * 100
    )


def masked_quantiles(
    values,
    mask,
    quantiles,
):
    selected = values[
        mask
        & np.isfinite(values)
    ]

    if selected.size == 0:
        return np.full(
            len(quantiles),
            np.nan,
            dtype=float,
        )

    return np.percentile(
        selected,
        quantiles,
    ).astype(float)


def circular_mean_degrees(
    values,
    mask,
):
    selected = values[
        mask
        & np.isfinite(values)
    ]

    if selected.size == 0:
        return np.nan, np.nan

    radians = np.deg2rad(
        selected
    )

    mean_sine = np.mean(
        np.sin(radians)
    )

    mean_cosine = np.mean(
        np.cos(radians)
    )

    angle = (
        np.rad2deg(
            np.arctan2(
                mean_sine,
                mean_cosine,
            )
        )
        + 360
    ) % 360

    resultant_length = np.hypot(
        mean_sine,
        mean_cosine,
    )

    return (
        float(angle),
        float(resultant_length),
    )


def mode_and_unique_count(
    values,
    mask,
):
    selected = values[
        mask
        & np.isfinite(values)
    ]

    if selected.size == 0:
        return np.nan, 0

    selected = np.rint(
        selected
    ).astype(int)

    unique_values, counts = np.unique(
        selected,
        return_counts=True,
    )

    mode = unique_values[
        np.argmax(counts)
    ]

    return (
        int(mode),
        int(len(unique_values)),
    )


def profile_band_values(
    date,
    source_file,
    variable,
    values,
):
    selected = values[
        support_mask
    ]

    finite = selected[
        np.isfinite(selected)
    ]

    valid_range = BAND_META[
        variable
    ]["valid_range"]

    if finite.size == 0:
        return {
            "Date": date,
            "Source_file": source_file,
            "Variable": variable,
            "Valid_pct": 0.0,
            "Out_of_range_pct": np.nan,
            "Min": np.nan,
            "P01": np.nan,
            "Median": np.nan,
            "P99": np.nan,
            "Max": np.nan,
        }

    out_of_range = (
        (finite < valid_range[0])
        | (finite > valid_range[1])
    )

    p01, median, p99 = (
        np.percentile(
            finite,
            [1, 50, 99],
        )
    )

    return {
        "Date": date,
        "Source_file": source_file,
        "Variable": variable,
        "Valid_pct": (
            finite.size
            / selected.size
            * 100
        ),
        "Out_of_range_pct": (
            out_of_range.mean()
            * 100
        ),
        "Min": float(
            np.min(finite)
        ),
        "P01": float(p01),
        "Median": float(median),
        "P99": float(p99),
        "Max": float(
            np.max(finite)
        ),
    }


SPATIAL_KEYS = [
    "core_valid_count",
    "qf_good_count",
    "clear_count",
    "strict_count",
    "vza_sum",
    "vza_count",
    "radiance_sum",
    "radiance_count",
]


def initialise_spatial_accumulators(
    shape
):
    return {
        "core_valid_count": (
            np.zeros(
                shape,
                dtype=np.uint32,
            )
        ),
        "qf_good_count": (
            np.zeros(
                shape,
                dtype=np.uint32,
            )
        ),
        "clear_count": (
            np.zeros(
                shape,
                dtype=np.uint32,
            )
        ),
        "strict_count": (
            np.zeros(
                shape,
                dtype=np.uint32,
            )
        ),
        "vza_sum": (
            np.zeros(
                shape,
                dtype=np.float64,
            )
        ),
        "vza_count": (
            np.zeros(
                shape,
                dtype=np.uint32,
            )
        ),
        "radiance_sum": (
            np.zeros(
                shape,
                dtype=np.float64,
            )
        ),
        "radiance_count": (
            np.zeros(
                shape,
                dtype=np.uint32,
            )
        ),
    }


support_hash = hashlib.sha1(
    support_mask
    .astype(np.uint8)
    .tobytes()
).hexdigest()[:12]


def cache_signature(
    file_path,
    dates,
):
    file_path = Path(file_path)
    file_stat = file_path.stat()

    payload = {
        "processing_version": (
            PROCESSING_VERSION
        ),
        "file": str(
            file_path.resolve()
        ),
        "size": file_stat.st_size,
        "mtime_ns": (
            file_stat.st_mtime_ns
        ),
        "dates": [
            pd.Timestamp(date).strftime(
                "%Y-%m-%d"
            )
            for date in dates
        ],
        "scaling_scheme": (
            SCALING_SCHEME
        ),
        "denominator": (
            OBSERVABILITY_DENOMINATOR
        ),
        "support_hash": support_hash,
        "profile_every": (
            PROFILE_EVERY_N_DAYS
        ),
    }

    return hashlib.sha1(
        json.dumps(
            payload,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()[:12]


def cache_paths(
    file_path,
    dates,
):
    safe_stem = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        Path(file_path).stem,
    )

    prefix = (
        CACHE_DIR
        / (
            f"{safe_stem}_"
            f"{cache_signature(file_path, dates)}"
        )
    )

    return {
        "daily": Path(
            f"{prefix}_daily.csv"
        ),
        "profile": Path(
            f"{prefix}_profiles.csv"
        ),
        "spatial": Path(
            f"{prefix}_spatial.npz"
        ),
    }


def save_file_cache(
    paths,
    daily_rows,
    profile_rows,
    spatial,
    processed_dates,
):
    (
        pd.DataFrame(daily_rows)
        .sort_values("Date")
        .to_csv(
            paths["daily"],
            index=False,
        )
    )

    pd.DataFrame(
        profile_rows
    ).to_csv(
        paths["profile"],
        index=False,
    )

    np.savez_compressed(
        paths["spatial"],
        processed_dates=np.array(
            [
                pd.Timestamp(
                    date
                ).strftime(
                    "%Y-%m-%d"
                )
                for date in sorted(
                    processed_dates
                )
            ],
            dtype="U10",
        ),
        **spatial,
    )


def load_file_cache(
    paths,
    expected_file_dates,
):
    if (
        FORCE_REPROCESS
        or not all(
            path.exists()
            for path in paths.values()
        )
    ):
        return None

    daily_cache = pd.read_csv(
        paths["daily"],
        parse_dates=["Date"],
    )

    profile_cache = pd.read_csv(
        paths["profile"],
        parse_dates=["Date"],
    )

    with np.load(
        paths["spatial"]
    ) as archive:
        if not all(
            key in archive
            for key in SPATIAL_KEYS
        ):
            return None

        spatial = {
            key: archive[key].copy()
            for key in SPATIAL_KEYS
        }

        spatial_dates = set(
            pd.to_datetime(
                archive[
                    "processed_dates"
                ].astype(str)
            )
        )

    daily_dates = set(
        pd.to_datetime(
            daily_cache["Date"]
        )
    )

    expected_file_dates = set(
        pd.to_datetime(
            expected_file_dates
        )
    )

    if (
        daily_dates != spatial_dates
        or not daily_dates.issubset(
            expected_file_dates
        )
    ):
        warnings.warn(
            "Ignoring inconsistent cache: "
            f"{paths['daily'].name}"
        )

        return None

    return (
        daily_cache,
        profile_cache,
        spatial,
        daily_dates,
    )

In [10]:
def parse_band_description(
    description,
    band_index,
):
    description = (
        str(description)
        if description is not None
        else ""
    )

    date_match = re.search(
        r"((?:19|20)\d{2})[_-]?(\d{2})[_-]?(\d{2})",
        description,
    )

    date = pd.NaT

    if date_match is not None:
        date = pd.Timestamp(
            year=int(date_match.group(1)),
            month=int(date_match.group(2)),
            day=int(date_match.group(3)),
        )

    variable = next(
        (
            band
            for band in sorted(
                EXPECTED_BANDS,
                key=len,
                reverse=True,
            )
            if band in description
        ),
        None,
    )

    return {
        "band_index": band_index,
        "description": description,
        "date": date,
        "variable": variable,
    }


def create_band_inventory(input_files):
    metadata_rows = []
    file_rows = []

    for file_path in input_files:
        with rasterio.open(file_path) as src:
            file_rows.append({
                "file": str(file_path),
                "width": src.width,
                "height": src.height,
                "band_count": src.count,
                "dtype": src.dtypes[0],
                "nodata": src.nodata,
                "crs": (
                    src.crs.to_string()
                    if src.crs is not None
                    else None
                ),
                "transform": tuple(src.transform)[:6],
                "bounds": tuple(src.bounds),
                "size_gb": (
                    file_path.stat().st_size
                    / 1024**3
                ),
            })

            for band_index in range(
                1,
                src.count + 1,
            ):
                description = (
                    src.descriptions[
                        band_index - 1
                    ]
                )

                if description is None:
                    description = (
                        src.tags(band_index)
                        .get("DESCRIPTION")
                    )

                parsed = parse_band_description(
                    description,
                    band_index,
                )

                parsed["file"] = str(file_path)
                metadata_rows.append(parsed)

    return (
        pd.DataFrame(metadata_rows),
        pd.DataFrame(file_rows),
    )


def to_uint16(values):
    integer_values = np.zeros(
        values.shape,
        dtype=np.uint16,
    )

    valid = np.isfinite(values)

    integer_values[valid] = (
        np.rint(values[valid])
        .astype(np.uint16)
    )

    return integer_values, valid


def decode_cloud_mask(values):
    integer_values, valid = to_uint16(
        values
    )

    return {
        "valid": valid,
        "day_night": (
            integer_values >> 0
        ) & 1,
        "land_water": (
            integer_values >> 1
        ) & 7,
        "mask_quality": (
            integer_values >> 4
        ) & 3,
        "cloud_confidence": (
            integer_values >> 6
        ) & 3,
        "shadow": (
            integer_values >> 8
        ) & 1,
        "cirrus": (
            integer_values >> 9
        ) & 1,
        "snow_ice": (
            integer_values >> 10
        ) & 1,
    }


def read_variable(
    sources,
    band_lookup,
    date,
    variable,
    out_shape=None,
):
    key = (
        pd.Timestamp(date),
        variable,
    )

    if key not in band_lookup:
        return None

    record = band_lookup[key]
    src = sources[record["file"]]

    read_kwargs = {}

    if out_shape is not None:
        read_kwargs.update({
            "out_shape": out_shape,
            "resampling": Resampling.nearest,
        })

    values = src.read(
        int(record["band_index"]),
        **read_kwargs,
    ).astype(np.float64)

    invalid = ~np.isfinite(values)

    invalid |= np.isclose(
        values,
        NO_DATA,
    )

    if src.nodata is not None:
        invalid |= np.isclose(
            values,
            src.nodata,
        )

    for fill_value in BAND_META[
        variable
    ]["fill_values"]:
        invalid |= np.isclose(
            values,
            fill_value,
        )

    values[invalid] = np.nan

    if DATA_ARE_RAW_GEE_VALUES:
        values *= BAND_META[
            variable
        ]["scale"]

    return values


band_metadata, file_inventory = (
    create_band_inventory(INPUT_FILES)
)

display(file_inventory.round(4))


unparsed_bands = band_metadata[
    band_metadata["date"].isna()
    | band_metadata["variable"].isna()
]

if not unparsed_bands.empty:
    display(
        unparsed_bands[
            [
                "file",
                "band_index",
                "description",
            ]
        ].head(30)
    )

    raise ValueError(
        "Some GeoTIFF bands could not be parsed. "
        "Inspect their descriptions before continuing."
    )


grid_signatures = set(
    zip(
        file_inventory["width"],
        file_inventory["height"],
        file_inventory["crs"],
        file_inventory["transform"].astype(str),
        file_inventory["bounds"].astype(str),
    )
)

if len(grid_signatures) != 1:
    raise ValueError(
        "Input GeoTIFFs do not share the same grid. "
        "Do not combine spatially tiled exports with "
        "date-batched exports in this workflow."
    )


duplicate_keys = band_metadata.duplicated(
    subset=[
        "date",
        "variable",
    ],
    keep=False,
)

if duplicate_keys.any():
    warnings.warn(
        "Duplicate date-variable bands were found. "
        "The first occurrence will be used."
    )

    display(
        band_metadata.loc[
            duplicate_keys,
            [
                "file",
                "date",
                "variable",
                "band_index",
            ],
        ].sort_values(
            [
                "date",
                "variable",
                "file",
            ]
        )
    )


selected_metadata = (
    band_metadata
    .sort_values(
        [
            "date",
            "variable",
            "file",
        ]
    )
    .drop_duplicates(
        subset=[
            "date",
            "variable",
        ],
        keep="first",
    )
)


band_lookup = (
    selected_metadata
    .set_index(
        [
            "date",
            "variable",
        ]
    )[
        [
            "file",
            "band_index",
        ]
    ]
    .to_dict("index")
)


available_dates = sorted(
    selected_metadata["date"].unique()
)

expected_dates = pd.date_range(
    min(available_dates),
    max(available_dates),
    freq="D",
)


date_inventory_rows = []

for date in expected_dates:
    available = set(
        selected_metadata.loc[
            selected_metadata["date"] == date,
            "variable",
        ]
    )

    missing = [
        band
        for band in EXPECTED_BANDS
        if band not in available
    ]

    date_inventory_rows.append({
        "Date": date,
        "Available_bands": len(available),
        "Missing_bands": ", ".join(missing),
        "Complete_band_set": len(missing) == 0,
    })


date_inventory_df = pd.DataFrame(
    date_inventory_rows
)

display(
    date_inventory_df[
        ~date_inventory_df[
            "Complete_band_set"
        ]
    ]
)

print(
    "Expected calendar days:",
    len(expected_dates),
)

print(
    "Days with all selected bands:",
    int(
        date_inventory_df[
            "Complete_band_set"
        ].sum()
    ),
)

print(
    "Parsed GeoTIFF bands:",
    len(selected_metadata),
)

,file,width,height,band_count,dtype,nodata,crs,transform,bounds,size_gb
0,../datasets/vnp46a1/Haiyan_VNP46A1_KeyBands_Pr...,473,674,1440,float32,None,EPSG:4326,"(0.00416666666, 0.0, 123.99999951360002, 0.0, ...","(123.99999951360002, 9.891666762839996, 125.97...",0.3879


,Date,Available_bands,Missing_bands,Complete_band_set


Expected calendar days: 180
Days with all selected bands: 180
Parsed GeoTIFF bands: 1440


In [16]:
def masked_quantiles(
    array,
    mask,
    quantiles,
):
    values = array[
        mask & np.isfinite(array)
    ]

    if values.size == 0:
        return np.full(
            len(quantiles),
            np.nan,
            dtype=float,
        )

    return np.percentile(
        values,
        quantiles,
    )


complete_band_by_date = (
    date_inventory_df
    .set_index("Date")[
        "Complete_band_set"
    ]
    .to_dict()
)

In [ ]:
# from tqdm.auto import tqdm


# def percentage(mask, denominator_mask):
#     denominator = np.count_nonzero(
#         denominator_mask
#     )

#     if denominator == 0:
#         return np.nan

#     return (
#         np.count_nonzero(
#             mask & denominator_mask
#         )
#         / denominator
#         * 100
#     )


# def masked_quantiles(
#     values,
#     mask,
#     quantiles,
# ):
#     selected = values[
#         mask & np.isfinite(values)
#     ]

#     if selected.size == 0:
#         return np.full(
#             len(quantiles),
#             np.nan,
#             dtype=float,
#         )

#     return np.percentile(
#         selected,
#         quantiles,
#     ).astype(float)


# def circular_mean_degrees(
#     values,
#     mask,
# ):
#     selected = values[
#         mask & np.isfinite(values)
#     ]

#     if selected.size == 0:
#         return np.nan, np.nan

#     radians = np.deg2rad(selected)

#     mean_sine = np.mean(
#         np.sin(radians)
#     )

#     mean_cosine = np.mean(
#         np.cos(radians)
#     )

#     angle = (
#         np.rad2deg(
#             np.arctan2(
#                 mean_sine,
#                 mean_cosine,
#             )
#         )
#         + 360
#     ) % 360

#     resultant_length = np.sqrt(
#         mean_sine**2
#         + mean_cosine**2
#     )

#     return (
#         float(angle),
#         float(resultant_length),
#     )


# def mode_and_unique_count(
#     values,
#     mask,
# ):
#     selected = values[
#         mask & np.isfinite(values)
#     ]

#     if selected.size == 0:
#         return np.nan, 0

#     selected = np.rint(
#         selected
#     ).astype(int)

#     unique_values, counts = np.unique(
#         selected,
#         return_counts=True,
#     )

#     mode = unique_values[
#         np.argmax(counts)
#     ]

#     return int(mode), len(unique_values)


# def profile_band_values(
#     date,
#     variable,
#     values,
#     support_mask,
# ):
#     selected = values[support_mask]
#     finite = selected[
#         np.isfinite(selected)
#     ]

#     valid_range = BAND_META[
#         variable
#     ]["valid_range"]

#     if finite.size == 0:
#         return {
#             "Date": date,
#             "Variable": variable,
#             "Valid_pct": 0.0,
#             "Out_of_range_pct": np.nan,
#             "Min": np.nan,
#             "P01": np.nan,
#             "Median": np.nan,
#             "P99": np.nan,
#             "Max": np.nan,
#         }

#     out_of_range = (
#         (finite < valid_range[0])
#         | (finite > valid_range[1])
#     )

#     p01, median, p99 = np.percentile(
#         finite,
#         [1, 50, 99],
#     )

#     return {
#         "Date": date,
#         "Variable": variable,
#         "Valid_pct": (
#             finite.size
#             / selected.size
#             * 100
#         ),
#         "Out_of_range_pct": (
#             out_of_range.mean()
#             * 100
#         ),
#         "Min": float(np.min(finite)),
#         "P01": float(p01),
#         "Median": float(median),
#         "P99": float(p99),
#         "Max": float(np.max(finite)),
#     }


# complete_band_by_date = (
#     date_inventory_df
#     .set_index("Date")[
#         "Complete_band_set"
#     ]
#     .astype(bool)
#     .to_dict()
# )


# complete_dates = sorted([
#     date
#     for date, complete
#     in complete_band_by_date.items()
#     if complete
# ])

# if not complete_dates:
#     raise ValueError(
#         "No date contains all eight selected VNP46A1 bands."
#     )


# daily_rows = []
# band_profile_rows = []


# with ExitStack() as stack:
#     sources = {
#         str(path): stack.enter_context(
#             rasterio.open(path)
#         )
#         for path in INPUT_FILES
#     }

#     reference_date = complete_dates[0]

#     reference_cloud_qf = read_variable(
#         sources,
#         band_lookup,
#         reference_date,
#         "QF_Cloud_Mask",
#     )

#     reference_cloud = decode_cloud_mask(
#         reference_cloud_qf
#     )

#     reference_roi_mask = (
#         reference_cloud["valid"]
#     )

#     reference_land_mask = (
#         reference_roi_mask
#         & np.isin(
#             reference_cloud["land_water"],
#             [0, 1, 5],
#         )
#     )

#     if (
#         OBSERVABILITY_DENOMINATOR
#         == "land_and_coast"
#     ):
#         support_mask = reference_land_mask
#     elif (
#         OBSERVABILITY_DENOMINATOR
#         == "all_roi"
#     ):
#         support_mask = reference_roi_mask
#     else:
#         raise ValueError(
#             "OBSERVABILITY_DENOMINATOR must be "
#             "'land_and_coast' or 'all_roi'."
#         )

#     if np.count_nonzero(support_mask) == 0:
#         raise ValueError(
#             "The reference denominator contains no pixels."
#         )

#     for date in tqdm(
#         expected_dates,
#         total=len(expected_dates),
#         desc="VNP46A1 daily diagnostics",
#         unit="day",
#         dynamic_ncols=True,
#         mininterval=0.5,
#     ):
#         if not complete_band_by_date.get(
#             date,
#             False,
#         ):
#             daily_rows.append({
#                 "Date": date,
#                 "Complete_band_set": False,
#             })
#             continue

#         arrays = {
#             variable: read_variable(
#                 sources,
#                 band_lookup,
#                 date,
#                 variable,
#             )
#             for variable in EXPECTED_BANDS
#         }

#         for variable, values in arrays.items():
#             band_profile_rows.append(
#                 profile_band_values(
#                     date,
#                     variable,
#                     values,
#                     support_mask,
#                 )
#             )

#         radiance = arrays[
#             "DNB_At_Sensor_Radiance_500m"
#         ]

#         sensor_zenith = arrays[
#             "Sensor_Zenith"
#         ]

#         sensor_azimuth = arrays[
#             "Sensor_Azimuth"
#         ]

#         utc_time = arrays[
#             "UTC_Time"
#         ]

#         granule = arrays[
#             "Granule"
#         ]

#         moon_illumination = arrays[
#             "Moon_Illumination_Fraction"
#         ]

#         cloud = decode_cloud_mask(
#             arrays["QF_Cloud_Mask"]
#         )

#         qf_dnb, qf_dnb_valid = to_uint16(
#             arrays["QF_DNB"]
#         )

#         core_valid = (
#             support_mask
#             & np.isfinite(radiance)
#             & cloud["valid"]
#             & qf_dnb_valid
#         )

#         source_valid = (
#             support_mask
#             & np.logical_and.reduce([
#                 np.isfinite(
#                     arrays[variable]
#                 )
#                 for variable in EXPECTED_BANDS
#             ])
#         )

#         night = (
#             cloud["valid"]
#             & (
#                 cloud["day_night"] == 0
#             )
#         )

#         confident_clear = (
#             cloud["valid"]
#             & (
#                 cloud[
#                     "cloud_confidence"
#                 ] == 0
#             )
#         )

#         probably_clear = (
#             cloud["valid"]
#             & (
#                 cloud[
#                     "cloud_confidence"
#                 ] == 1
#             )
#         )

#         probably_cloudy = (
#             cloud["valid"]
#             & (
#                 cloud[
#                     "cloud_confidence"
#                 ] == 2
#             )
#         )

#         confident_cloudy = (
#             cloud["valid"]
#             & (
#                 cloud[
#                     "cloud_confidence"
#                 ] == 3
#             )
#         )

#         medium_high_cloud_mask = (
#             cloud["valid"]
#             & (
#                 cloud[
#                     "mask_quality"
#                 ] >= 2
#             )
#         )

#         no_shadow = (
#             cloud["valid"]
#             & (
#                 cloud["shadow"] == 0
#             )
#         )

#         no_cirrus = (
#             cloud["valid"]
#             & (
#                 cloud["cirrus"] == 0
#             )
#         )

#         no_snow = (
#             cloud["valid"]
#             & (
#                 cloud["snow_ice"] == 0
#             )
#         )

#         dnb_good = (
#             qf_dnb_valid
#             & (
#                 qf_dnb == 0
#             )
#         )

#         strict_observable = (
#             core_valid
#             & night
#             & medium_high_cloud_mask
#             & confident_clear
#             & no_shadow
#             & no_cirrus
#             & no_snow
#             & dnb_good
#         )

#         inclusive_observable = (
#             core_valid
#             & night
#             & medium_high_cloud_mask
#             & (
#                 cloud[
#                     "cloud_confidence"
#                 ] <= 1
#             )
#             & no_shadow
#             & no_cirrus
#             & no_snow
#             & dnb_good
#         )

#         acquisition_mask = (
#             core_valid
#             & night
#         )

#         current_background_valid = (
#             support_mask
#             & cloud["valid"]
#         )

#         background_change = (
#             current_background_valid
#             & (
#                 cloud["land_water"]
#                 != reference_cloud[
#                     "land_water"
#                 ]
#             )
#         )

#         radiance_all_median = (
#             masked_quantiles(
#                 radiance,
#                 core_valid,
#                 [50],
#             )[0]
#         )

#         radiance_quantiles = (
#             masked_quantiles(
#                 radiance,
#                 strict_observable,
#                 [25, 50, 75, 95],
#             )
#         )

#         vza_quantiles = (
#             masked_quantiles(
#                 np.abs(sensor_zenith),
#                 acquisition_mask,
#                 [25, 50, 75, 90],
#             )
#         )

#         utc_quantiles = (
#             masked_quantiles(
#                 utc_time,
#                 acquisition_mask,
#                 [25, 50, 75],
#             )
#         )

#         moon_median = (
#             masked_quantiles(
#                 moon_illumination,
#                 acquisition_mask,
#                 [50],
#             )[0]
#         )

#         azimuth_mean, azimuth_resultant = (
#             circular_mean_degrees(
#                 sensor_azimuth,
#                 acquisition_mask,
#             )
#         )

#         granule_mode, granule_count = (
#             mode_and_unique_count(
#                 granule,
#                 acquisition_mask,
#             )
#         )

#         row = {
#             "Date": date,
#             "Complete_band_set": True,
#             "Source_valid_pct": percentage(
#                 source_valid,
#                 support_mask,
#             ),
#             "Core_valid_pct": percentage(
#                 core_valid,
#                 support_mask,
#             ),
#             "Night_pct": percentage(
#                 night,
#                 support_mask,
#             ),
#             "Cloud_mask_medium_high_pct": (
#                 percentage(
#                     medium_high_cloud_mask,
#                     support_mask,
#                 )
#             ),
#             "Confident_clear_pct": percentage(
#                 confident_clear,
#                 support_mask,
#             ),
#             "Probably_clear_pct": percentage(
#                 probably_clear,
#                 support_mask,
#             ),
#             "Probably_cloudy_pct": percentage(
#                 probably_cloudy,
#                 support_mask,
#             ),
#             "Confident_cloudy_pct": percentage(
#                 confident_cloudy,
#                 support_mask,
#             ),
#             "Cloudy_pct": percentage(
#                 probably_cloudy
#                 | confident_cloudy,
#                 support_mask,
#             ),
#             "Shadow_pct": percentage(
#                 cloud["valid"]
#                 & (
#                     cloud["shadow"] == 1
#                 ),
#                 support_mask,
#             ),
#             "Cirrus_pct": percentage(
#                 cloud["valid"]
#                 & (
#                     cloud["cirrus"] == 1
#                 ),
#                 support_mask,
#             ),
#             "Snow_ice_pct": percentage(
#                 cloud["valid"]
#                 & (
#                     cloud["snow_ice"] == 1
#                 ),
#                 support_mask,
#             ),
#             "QF_DNB_good_pct": percentage(
#                 dnb_good,
#                 support_mask,
#             ),
#             "Strict_observable_pct": (
#                 percentage(
#                     strict_observable,
#                     support_mask,
#                 )
#             ),
#             "Inclusive_observable_pct": (
#                 percentage(
#                     inclusive_observable,
#                     support_mask,
#                 )
#             ),
#             "Strict_observable_pixels": (
#                 np.count_nonzero(
#                     strict_observable
#                 )
#             ),
#             "Land_water_change_pct": (
#                 percentage(
#                     background_change,
#                     support_mask,
#                 )
#             ),
#             "Radiance_all_median": (
#                 radiance_all_median
#             ),
#             "Radiance_strict_p25": (
#                 radiance_quantiles[0]
#             ),
#             "Radiance_strict_median": (
#                 radiance_quantiles[1]
#             ),
#             "Radiance_strict_p75": (
#                 radiance_quantiles[2]
#             ),
#             "Radiance_strict_p95": (
#                 radiance_quantiles[3]
#             ),
#             "Sensor_zenith_abs_p25": (
#                 vza_quantiles[0]
#             ),
#             "Sensor_zenith_abs_median": (
#                 vza_quantiles[1]
#             ),
#             "Sensor_zenith_abs_p75": (
#                 vza_quantiles[2]
#             ),
#             "Sensor_zenith_abs_p90": (
#                 vza_quantiles[3]
#             ),
#             "Sensor_azimuth_circular_mean": (
#                 azimuth_mean
#             ),
#             "Sensor_azimuth_resultant": (
#                 azimuth_resultant
#             ),
#             "UTC_time_p25": (
#                 utc_quantiles[0]
#             ),
#             "UTC_time_median": (
#                 utc_quantiles[1]
#             ),
#             "UTC_time_p75": (
#                 utc_quantiles[2]
#             ),
#             "Moon_illumination_pct": (
#                 moon_median
#             ),
#             "Granule_mode": granule_mode,
#             "Granule_unique_count": (
#                 granule_count
#             ),
#         }

#         for flag_name, flag_value in (
#             QF_DNB_FLAGS.items()
#         ):
#             flag_mask = (
#                 qf_dnb_valid
#                 & (
#                     (
#                         qf_dnb
#                         & flag_value
#                     ) > 0
#                 )
#             )

#             row[
#                 f"QF_{flag_name}_pct"
#             ] = percentage(
#                 flag_mask,
#                 support_mask,
#             )

#         daily_rows.append(row)


# daily_df = (
#     pd.DataFrame(daily_rows)
#     .sort_values("Date")
#     .reset_index(drop=True)
# )

# daily_df["Days_from_event"] = (
#     daily_df["Date"]
#     - EVENT_DATE
# ).dt.days


# band_profile_df = (
#     pd.DataFrame(band_profile_rows)
#     .sort_values(
#         [
#             "Date",
#             "Variable",
#         ]
#     )
#     .reset_index(drop=True)
# )


# band_profile_summary = (
#     band_profile_df
#     .groupby(
#         "Variable",
#         as_index=False,
#     )
#     .agg(
#         Dates=("Date", "nunique"),
#         Median_valid_pct=(
#             "Valid_pct",
#             "median",
#         ),
#         Minimum_valid_pct=(
#             "Valid_pct",
#             "min",
#         ),
#         Maximum_out_of_range_pct=(
#             "Out_of_range_pct",
#             "max",
#         ),
#         Minimum=("Min", "min"),
#         Median=("Median", "median"),
#         Maximum=("Max", "max"),
#     )
# )


# if SAVE_OUTPUTS:
#     daily_df.to_csv(
#         OUTPUT_TABLE_DIR
#         / "vnp46a1_daily_diagnostics.csv",
#         index=False,
#     )

#     band_profile_df.to_csv(
#         OUTPUT_TABLE_DIR
#         / "vnp46a1_band_profiles_by_date.csv",
#         index=False,
#     )

#     band_profile_summary.to_csv(
#         OUTPUT_TABLE_DIR
#         / "vnp46a1_band_profile_summary.csv",
#         index=False,
#     )


# display(
#     band_profile_summary.round(4)
# )

# display(
#     daily_df.head().round(3)
# )

VNP46A1 daily diagnostics: 100%|██████████| 180/180 [1:13:12<00:00, 24.40s/day]


NameError: name 'OUTPUT_TABLE_DIR' is not defined

In [29]:
display(
    band_profile_summary.round(4)
)

display(
    daily_df.head().round(3)
)

,Variable,Dates,Median_valid_pct,Minimum_valid_pct,Maximum_out_of_range_pct,Minimum,Median,Maximum
0,DNB_At_Sensor_Radiance_500m,180,100.0,100.0,0.0,0.0000,0.0500,17.9700
1,Granule,180,100.0,100.0,0.0,0.0000,1.0000,3.0000
2,Moon_Illumination_Fraction,180,100.0,100.0,0.0,0.0007,0.4444,0.9989
3,QF_Cloud_Mask,180,100.0,100.0,0.0,0.0000,738.0000,762.0000
4,QF_DNB,180,100.0,100.0,0.0,0.0000,0.0000,0.0000
5,Sensor_Azimuth,180,100.0,100.0,0.0,-1.7994,0.0912,1.7992
6,Sensor_Zenith,180,100.0,100.0,0.0,0.0002,0.4389,0.6635
7,UTC_Time,180,100.0,100.0,0.0,16.3102,17.2115,18.1146


,Date,Complete_band_set,Source_valid_pct,Core_valid_pct,Night_pct,Cloud_mask_medium_high_pct,Confident_clear_pct,Probably_clear_pct,Probably_cloudy_pct,Confident_cloudy_pct,...,QF_Substitute_calibration_pct,QF_Out_of_range_pct,QF_Saturation_pct,QF_Temperature_not_nominal_pct,QF_Stray_light_pct,QF_Bowtie_deleted_pct,QF_Missing_EV_pct,QF_Calibration_failure_pct,QF_Dead_detector_pct,Days_from_event
0,2013-05-12,True,100.0,100.0,100.0,100.0,0.458,1.028,0.380,98.134,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-180
1,2013-05-13,True,100.0,100.0,100.0,100.0,0.000,0.000,0.000,100.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-179
2,2013-05-14,True,100.0,100.0,100.0,100.0,10.597,5.463,1.594,82.347,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-178
3,2013-05-15,True,100.0,100.0,100.0,100.0,92.693,2.421,0.784,4.102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-177
4,2013-05-16,True,100.0,100.0,100.0,100.0,91.665,1.369,0.303,6.664,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-176


In [30]:
if SAVE_OUTPUTS:
    daily_df.to_csv(
        OUTPUT_TABLE_DIR
        / "vnp46a1_daily_diagnostics.csv",
        index=False,
    )

    band_profile_df.to_csv(
        OUTPUT_TABLE_DIR
        / "vnp46a1_band_profiles_by_date.csv",
        index=False,
    )

    band_profile_summary.to_csv(
        OUTPUT_TABLE_DIR
        / "vnp46a1_band_profile_summary.csv",
        index=False,
    )

In [31]:
daily_df = pd.read_csv(
    OUTPUT_TABLE_DIR
    / "vnp46a1_daily_diagnostics.csv",
    parse_dates=["Date"],
)

band_profile_df = pd.read_csv(
    OUTPUT_TABLE_DIR
    / "vnp46a1_band_profiles_by_date.csv",
    parse_dates=["Date"],
)

band_profile_summary = pd.read_csv(
    OUTPUT_TABLE_DIR
    / "vnp46a1_band_profile_summary.csv"
)


daily_df["Days_from_event"] = (
    daily_df["Date"]
    - EVENT_DATE
).dt.days


display(
    band_profile_summary.round(4)
)

display(
    daily_df.head().round(3)
)

,Variable,Dates,Median_valid_pct,Minimum_valid_pct,Maximum_out_of_range_pct,Minimum,Median,Maximum
0,DNB_At_Sensor_Radiance_500m,180,100.0,100.0,0.0,0.0000,0.0500,17.9700
1,Granule,180,100.0,100.0,0.0,0.0000,1.0000,3.0000
2,Moon_Illumination_Fraction,180,100.0,100.0,0.0,0.0007,0.4444,0.9989
3,QF_Cloud_Mask,180,100.0,100.0,0.0,0.0000,738.0000,762.0000
4,QF_DNB,180,100.0,100.0,0.0,0.0000,0.0000,0.0000
5,Sensor_Azimuth,180,100.0,100.0,0.0,-1.7994,0.0912,1.7992
6,Sensor_Zenith,180,100.0,100.0,0.0,0.0002,0.4389,0.6635
7,UTC_Time,180,100.0,100.0,0.0,16.3102,17.2115,18.1146


,Date,Complete_band_set,Source_valid_pct,Core_valid_pct,Night_pct,Cloud_mask_medium_high_pct,Confident_clear_pct,Probably_clear_pct,Probably_cloudy_pct,Confident_cloudy_pct,...,QF_Substitute_calibration_pct,QF_Out_of_range_pct,QF_Saturation_pct,QF_Temperature_not_nominal_pct,QF_Stray_light_pct,QF_Bowtie_deleted_pct,QF_Missing_EV_pct,QF_Calibration_failure_pct,QF_Dead_detector_pct,Days_from_event
0,2013-05-12,True,100.0,100.0,100.0,100.0,0.458,1.028,0.380,98.134,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-180
1,2013-05-13,True,100.0,100.0,100.0,100.0,0.000,0.000,0.000,100.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-179
2,2013-05-14,True,100.0,100.0,100.0,100.0,10.597,5.463,1.594,82.347,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-178
3,2013-05-15,True,100.0,100.0,100.0,100.0,92.693,2.421,0.784,4.102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-177
4,2013-05-16,True,100.0,100.0,100.0,100.0,91.665,1.369,0.303,6.664,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-176


In [32]:
def apply_figure_style(
    fig,
    width=1300,
    height=700,
    top=115,
    right=60,
):
    fig.update_layout(
        template="plotly_white",
        width=width,
        height=height,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        margin=dict(
            l=85,
            r=right,
            t=top,
            b=75,
        ),
        font=dict(
            family="Arial",
            size=15,
            color="#222222",
        ),
        title=dict(
            x=0.02,
            xanchor="left",
            font=dict(size=22),
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.01,
            xanchor="left",
            x=0.01,
            bgcolor="rgba(255,255,255,0.85)",
        ),
        hovermode="closest",
    )

    fig.update_xaxes(
        showline=True,
        linewidth=1,
        linecolor="#444444",
        mirror=False,
        gridcolor="rgba(0,0,0,0.08)",
    )

    fig.update_yaxes(
        showline=True,
        linewidth=1,
        linecolor="#444444",
        mirror=False,
        gridcolor="rgba(0,0,0,0.08)",
    )

    return fig


def add_haiyan_marker(
    fig,
    annotation=True,
):
    fig.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line_dash="dash",
        line_color="#1f77b4",
        line_width=2,
    )

    if annotation:
        fig.add_annotation(
            x=EVENT_DATE.to_pydatetime(),
            y=1.02,
            yref="paper",
            text="Haiyan · 8 Nov 2013",
            showarrow=False,
            xanchor="left",
            yanchor="bottom",
            font=dict(
                size=13,
                color="#1f77b4",
            ),
            bgcolor="rgba(255,255,255,0.85)",
        )

    return fig


def export_figure(
    fig,
    filename,
):
    if not SAVE_OUTPUTS:
        return

    fig.write_html(
        OUTPUT_FIGURE_DIR
        / f"{filename}.html"
    )

    if SAVE_PNG:
        try:
            fig.write_image(
                OUTPUT_FIGURE_DIR
                / f"{filename}.png",
                scale=2,
            )
        except Exception as error:
            warnings.warn(
                f"PNG export skipped: {error}"
            )


plot_df = daily_df[
    daily_df["Complete_band_set"]
].copy()

In [33]:
fig_observability = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=(
        "Usable spatial coverage",
        "Cloud-confidence composition",
        "Supporting quality conditions",
    ),
)


for column, name, color, width in [
    (
        "Source_valid_pct",
        "All eight bands valid",
        "#7f7f7f",
        1.8,
    ),
    (
        "Inclusive_observable_pct",
        "Inclusive observable",
        "#ff7f0e",
        2.0,
    ),
    (
        "Strict_observable_pct",
        "Strict observable",
        "#1f77b4",
        2.8,
    ),
]:
    fig_observability.add_trace(
        go.Scatter(
            x=plot_df["Date"],
            y=plot_df[column],
            mode="lines",
            line=dict(
                color=color,
                width=width,
            ),
            name=name,
            connectgaps=False,
        ),
        row=1,
        col=1,
    )


for column, name, color in [
    (
        "Confident_clear_pct",
        "Confident clear",
        "#2ca02c",
    ),
    (
        "Probably_clear_pct",
        "Probably clear",
        "#98df8a",
    ),
    (
        "Probably_cloudy_pct",
        "Probably cloudy",
        "#ffbb78",
    ),
    (
        "Confident_cloudy_pct",
        "Confident cloudy",
        "#d62728",
    ),
]:
    fig_observability.add_trace(
        go.Scatter(
            x=plot_df["Date"],
            y=plot_df[column],
            mode="lines",
            line=dict(
                color=color,
                width=0.7,
            ),
            stackgroup="cloud",
            name=name,
            connectgaps=False,
        ),
        row=2,
        col=1,
    )


for column, name, color in [
    (
        "QF_DNB_good_pct",
        "QF_DNB = 0",
        "#9467bd",
    ),
    (
        "Cloud_mask_medium_high_pct",
        "Medium/high cloud-mask quality",
        "#17becf",
    ),
    (
        "Night_pct",
        "Night classification",
        "#4d4d4d",
    ),
]:
    fig_observability.add_trace(
        go.Scatter(
            x=plot_df["Date"],
            y=plot_df[column],
            mode="lines",
            line=dict(
                color=color,
                width=2,
            ),
            name=name,
            connectgaps=False,
        ),
        row=3,
        col=1,
    )


for row in [1, 2, 3]:
    fig_observability.update_yaxes(
        title_text="Land pixels (%)",
        range=[0, 100],
        row=row,
        col=1,
    )


fig_observability.update_xaxes(
    title_text="Date",
    range=[
        plot_df["Date"].min(),
        max(
            plot_df["Date"].max()
            + pd.Timedelta(days=5),
            EVENT_DATE
            + pd.Timedelta(days=5),
        ),
    ],
    row=3,
    col=1,
)


fig_observability.update_layout(
    title=(
        "VNP46A1 Observability before and around Haiyan"
        "<br><sup>"
        "Strict: night · medium/high cloud-mask quality · "
        "confident clear · no shadow/cirrus/snow · QF_DNB = 0"
        "</sup>"
    )
)

add_haiyan_marker(
    fig_observability
)

apply_figure_style(
    fig_observability,
    width=1350,
    height=950,
    top=135,
)

fig_observability.show()

export_figure(
    fig_observability,
    "fig_01_vnp46a1_observability",
)

In [34]:
fig_geometry = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=(
        "Absolute sensor zenith angle",
        "Moon illumination",
        "Acquisition time",
    ),
)


fig_geometry.add_trace(
    go.Scatter(
        x=plot_df["Date"],
        y=plot_df[
            "Sensor_zenith_abs_p25"
        ],
        mode="lines",
        line=dict(
            width=0,
            color="rgba(31,119,180,0)",
        ),
        showlegend=False,
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)

fig_geometry.add_trace(
    go.Scatter(
        x=plot_df["Date"],
        y=plot_df[
            "Sensor_zenith_abs_p75"
        ],
        mode="lines",
        line=dict(
            width=0,
            color="rgba(31,119,180,0)",
        ),
        fill="tonexty",
        fillcolor="rgba(31,119,180,0.18)",
        name="Spatial IQR",
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)

fig_geometry.add_trace(
    go.Scatter(
        x=plot_df["Date"],
        y=plot_df[
            "Sensor_zenith_abs_median"
        ],
        mode="lines+markers",
        line=dict(
            color="#1f77b4",
            width=2,
        ),
        marker=dict(size=5),
        name="Median |VZA|",
    ),
    row=1,
    col=1,
)

fig_geometry.add_trace(
    go.Scatter(
        x=plot_df["Date"],
        y=plot_df[
            "Sensor_zenith_abs_p90"
        ],
        mode="lines",
        line=dict(
            color="#1f77b4",
            width=1.3,
            dash="dot",
        ),
        name="90th percentile |VZA|",
    ),
    row=1,
    col=1,
)


fig_geometry.add_trace(
    go.Scatter(
        x=plot_df["Date"],
        y=plot_df[
            "Moon_illumination_pct"
        ],
        mode="lines",
        line=dict(
            color="#9467bd",
            width=2.2,
        ),
        name="Moon illumination",
    ),
    row=2,
    col=1,
)


fig_geometry.add_trace(
    go.Scatter(
        x=plot_df["Date"],
        y=plot_df["UTC_time_median"],
        mode="lines+markers",
        line=dict(
            color="#ff7f0e",
            width=2,
        ),
        marker=dict(size=5),
        name="Median UTC time",
    ),
    row=3,
    col=1,
)


fig_geometry.update_yaxes(
    title_text="|VZA| (degrees)",
    row=1,
    col=1,
)

fig_geometry.update_yaxes(
    title_text="Illumination (%)",
    range=[0, 100],
    row=2,
    col=1,
)

fig_geometry.update_yaxes(
    title_text="UTC hour",
    range=[0, 24],
    row=3,
    col=1,
)

fig_geometry.update_xaxes(
    title_text="Date",
    range=[
        plot_df["Date"].min(),
        max(
            plot_df["Date"].max()
            + pd.Timedelta(days=5),
            EVENT_DATE
            + pd.Timedelta(days=5),
        ),
    ],
    row=3,
    col=1,
)


fig_geometry.update_layout(
    title=(
        "VNP46A1 Acquisition-Condition Diagnostics"
        "<br><sup>"
        "Viewing geometry and lunar illumination are "
        "potential controls on at-sensor radiance"
        "</sup>"
    )
)

add_haiyan_marker(
    fig_geometry
)

apply_figure_style(
    fig_geometry,
    width=1350,
    height=900,
    top=125,
)

fig_geometry.show()

export_figure(
    fig_geometry,
    "fig_02_vnp46a1_acquisition_conditions",
)

In [38]:
CORRELATION_COLUMNS = [
    "Strict_observable_pct",
    "Inclusive_observable_pct",
    "Cloudy_pct",
    "QF_DNB_good_pct",
    "Cloud_mask_medium_high_pct",
    "Sensor_zenith_abs_median",
    "UTC_time_median",
    "Moon_illumination_pct",
    "Radiance_strict_median",
    "Radiance_all_median",
]


CORRELATION_LABELS = {
    "Strict_observable_pct": (
        "Strict observable"
    ),
    "Inclusive_observable_pct": (
        "Inclusive observable"
    ),
    "Cloudy_pct": "Cloudy",
    "QF_DNB_good_pct": "QF_DNB = 0",
    "Cloud_mask_medium_high_pct": (
        "Cloud-mask quality"
    ),
    "Sensor_zenith_abs_median": (
        "Median |VZA|"
    ),
    "UTC_time_median": "UTC time",
    "Moon_illumination_pct": (
        "Moon illumination"
    ),
    "Radiance_strict_median": (
        "Strict radiance"
    ),
    "Radiance_all_median": (
        "All-valid radiance"
    ),
}


correlation_data = plot_df[
    CORRELATION_COLUMNS
].copy()


spearman_raw = correlation_data.corr(
    method="spearman"
)

pairwise_n_raw = (
    correlation_data
    .notna()
    .astype(int)
    .T
    @ correlation_data
    .notna()
    .astype(int)
)


spearman_df = spearman_raw.rename(
    index=CORRELATION_LABELS,
    columns=CORRELATION_LABELS,
)

pairwise_n_df = pairwise_n_raw.rename(
    index=CORRELATION_LABELS,
    columns=CORRELATION_LABELS,
)


if SAVE_OUTPUTS:
    spearman_df.to_csv(
        OUTPUT_TABLE_DIR
        / "vnp46a1_spearman_correlations.csv"
    )

    pairwise_n_df.to_csv(
        OUTPUT_TABLE_DIR
        / "vnp46a1_correlation_pairwise_n.csv"
    )


observability_correlation_rows = []

for column in CORRELATION_COLUMNS:
    if column == "Strict_observable_pct":
        continue

    observability_correlation_rows.append({
        "Variable": CORRELATION_LABELS[
            column
        ],
        "Spearman_rho": spearman_raw.loc[
            "Strict_observable_pct",
            column,
        ],
        "Pairwise_n": int(
            pairwise_n_raw.loc[
                "Strict_observable_pct",
                column,
            ]
        ),
    })


observability_correlations_df = (
    pd.DataFrame(
        observability_correlation_rows
    )
    .assign(
        Absolute_rho=lambda data: (
            data["Spearman_rho"].abs()
        )
    )
    .sort_values(
        "Absolute_rho",
        ascending=False,
    )
    .drop(columns="Absolute_rho")
)


display(
    observability_correlations_df.round(3)
)


fig_correlation = go.Figure(
    go.Heatmap(
        z=spearman_df.values,
        x=spearman_df.columns,
        y=spearman_df.index,
        zmin=-1,
        zmax=1,
        zmid=0,
        colorscale="RdBu_r",
        text=np.round(
            spearman_df.values,
            2,
        ),
        texttemplate="%{text:.2f}",
        colorbar=dict(
            title="Spearman ρ",
        ),
        hovertemplate=(
            "X: %{x}<br>"
            "Y: %{y}<br>"
            "ρ: %{z:.3f}"
            "<extra></extra>"
        ),
    )
)


fig_correlation.update_layout(
    title=(
        "VNP46A1 Observability and Acquisition Correlations"
        "<br><sup>"
        "Pairwise Spearman correlations; association "
        "does not establish a causal observation filter"
        "</sup>"
    )
)

fig_correlation.update_yaxes(
    autorange="reversed"
)

apply_figure_style(
    fig_correlation,
    width=1100,
    height=850,
    top=120,
    right=120,
)

fig_correlation.show()

export_figure(
    fig_correlation,
    "fig_03_vnp46a1_correlation_matrix",
)

,Variable,Spearman_rho,Pairwise_n
0,Inclusive observable,0.999,180
1,Cloudy,-0.994,180
8,All-valid radiance,-0.112,180
3,Cloud-mask quality,-0.100,180
5,UTC time,0.058,180
4,Median |VZA|,-0.049,180
6,Moon illumination,0.049,180
7,Strict radiance,0.041,137
2,QF_DNB = 0,NaN,180


In [39]:
def spearman_rho(
    data,
    x_column,
    y_column,
):
    subset = data[
        [
            x_column,
            y_column,
        ]
    ].dropna()

    if len(subset) < 3:
        return np.nan

    return subset.corr(
        method="spearman"
    ).iloc[0, 1]


OBSERVABILITY_DRIVERS = [
    (
        "Sensor_zenith_abs_median",
        "Median |VZA| (degrees)",
    ),
    (
        "Moon_illumination_pct",
        "Moon illumination (%)",
    ),
    (
        "UTC_time_median",
        "UTC acquisition time",
    ),
    (
        "QF_DNB_good_pct",
        "QF_DNB = 0 pixels (%)",
    ),
]


fig_observability_scatter = (
    make_subplots(
        rows=2,
        cols=2,
        horizontal_spacing=0.10,
        vertical_spacing=0.16,
    )
)


for position, (
    x_column,
    x_label,
) in enumerate(
    OBSERVABILITY_DRIVERS
):
    row = position // 2 + 1
    col = position % 2 + 1

    subset = plot_df[
        [
            "Date",
            "Days_from_event",
            x_column,
            "Strict_observable_pct",
            "Cloudy_pct",
        ]
    ].dropna()

    rho = spearman_rho(
        subset,
        x_column,
        "Strict_observable_pct",
    )

    fig_observability_scatter.add_trace(
        go.Scatter(
            x=subset[x_column],
            y=subset[
                "Strict_observable_pct"
            ],
            mode="markers",
            marker=dict(
                size=9,
                color=subset[
                    "Days_from_event"
                ],
                colorscale="RdBu_r",
                opacity=0.82,
                line=dict(
                    color="white",
                    width=0.7,
                ),
                showscale=(
                    position == 3
                ),
                colorbar=(
                    dict(
                        title="Days from Haiyan",
                        x=1.03,
                    )
                    if position == 3
                    else None
                ),
            ),
            customdata=np.column_stack([
                subset[
                    "Date"
                ].dt.strftime("%Y-%m-%d"),
                subset["Cloudy_pct"],
            ]),
            hovertemplate=(
                "Date: %{customdata[0]}<br>"
                f"{x_label}: %{{x:.2f}}<br>"
                "Strict observable: %{y:.2f}%<br>"
                "Cloudy: %{customdata[1]:.2f}%"
                "<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    fig_observability_scatter.add_annotation(
        x=0.03,
        y=0.95,
        xref=(
            "x domain"
            if position == 0
            else f"x{position + 1} domain"
        ),
        yref=(
            "y domain"
            if position == 0
            else f"y{position + 1} domain"
        ),
        text=(
            f"Spearman ρ = {rho:.2f}"
            if np.isfinite(rho)
            else "Spearman ρ = NA"
        ),
        showarrow=False,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.80)",
        font=dict(size=13),
    )

    fig_observability_scatter.update_xaxes(
        title_text=x_label,
        row=row,
        col=col,
    )

    fig_observability_scatter.update_yaxes(
        title_text=(
            "Strict observable (%)"
            if col == 1
            else None
        ),
        range=[0, 100],
        row=row,
        col=col,
    )


fig_observability_scatter.update_layout(
    title=(
        "Acquisition Conditions Associated with VNP46A1 Observability"
        "<br><sup>"
        "Strict observability remains primarily a quality-supported "
        "availability measure, not a recovery metric"
        "</sup>"
    )
)

apply_figure_style(
    fig_observability_scatter,
    width=1300,
    height=850,
    top=125,
    right=140,
)

fig_observability_scatter.show()

export_figure(
    fig_observability_scatter,
    "fig_04_vnp46a1_observability_drivers",
)

In [40]:
RADIANCE_DRIVERS = [
    (
        "Moon_illumination_pct",
        "Moon illumination (%)",
    ),
    (
        "Sensor_zenith_abs_median",
        "Median |VZA| (degrees)",
    ),
    (
        "UTC_time_median",
        "UTC acquisition time",
    ),
    (
        "Strict_observable_pct",
        "Strict observable (%)",
    ),
]


fig_radiance = make_subplots(
    rows=2,
    cols=2,
    horizontal_spacing=0.10,
    vertical_spacing=0.16,
)


for position, (
    x_column,
    x_label,
) in enumerate(
    RADIANCE_DRIVERS
):
    row = position // 2 + 1
    col = position % 2 + 1

    subset = plot_df[
        [
            "Date",
            "Days_from_event",
            x_column,
            "Radiance_strict_median",
            "Strict_observable_pct",
        ]
    ].dropna()

    rho = spearman_rho(
        subset,
        x_column,
        "Radiance_strict_median",
    )

    fig_radiance.add_trace(
        go.Scatter(
            x=subset[x_column],
            y=subset[
                "Radiance_strict_median"
            ],
            mode="markers",
            marker=dict(
                size=9,
                color=subset[
                    "Days_from_event"
                ],
                colorscale="RdBu_r",
                opacity=0.82,
                line=dict(
                    color="white",
                    width=0.7,
                ),
                showscale=(
                    position == 3
                ),
                colorbar=(
                    dict(
                        title="Days from Haiyan",
                        x=1.03,
                    )
                    if position == 3
                    else None
                ),
            ),
            customdata=np.column_stack([
                subset[
                    "Date"
                ].dt.strftime("%Y-%m-%d"),
                subset[
                    "Strict_observable_pct"
                ],
            ]),
            hovertemplate=(
                "Date: %{customdata[0]}<br>"
                f"{x_label}: %{{x:.2f}}<br>"
                "Median at-sensor radiance: %{y:.3f}<br>"
                "Strict observable: %{customdata[1]:.2f}%"
                "<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    fig_radiance.add_annotation(
        x=0.03,
        y=0.95,
        xref=(
            "x domain"
            if position == 0
            else f"x{position + 1} domain"
        ),
        yref=(
            "y domain"
            if position == 0
            else f"y{position + 1} domain"
        ),
        text=(
            f"Spearman ρ = {rho:.2f}"
            if np.isfinite(rho)
            else "Spearman ρ = NA"
        ),
        showarrow=False,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.80)",
        font=dict(size=13),
    )

    fig_radiance.update_xaxes(
        title_text=x_label,
        row=row,
        col=col,
    )

    fig_radiance.update_yaxes(
        title_text=(
            "Median radiance"
            if col == 1
            else None
        ),
        row=row,
        col=col,
    )


fig_radiance.update_layout(
    title=(
        "Sensitivity of Strict-Screened VNP46A1 At-Sensor Radiance"
        "<br><sup>"
        "VNP46A1 radiance remains uncorrected for atmosphere, "
        "moonlight and surface anisotropy"
        "</sup>"
    )
)

apply_figure_style(
    fig_radiance,
    width=1300,
    height=850,
    top=125,
    right=140,
)

fig_radiance.show()

export_figure(
    fig_radiance,
    "fig_05_vnp46a1_radiance_sensitivity",
)

DuplicateError: Expected unique column names, got:
- 'Strict_observable_pct' 2 times

In [41]:
qf_summary_rows = []


for flag_name in QF_DNB_FLAGS:
    column = f"QF_{flag_name}_pct"

    qf_summary_rows.append({
        "Flag": flag_name.replace(
            "_",
            " ",
        ),
        "Mean_pct": plot_df[
            column
        ].mean(),
        "Median_pct": plot_df[
            column
        ].median(),
        "Maximum_pct": plot_df[
            column
        ].max(),
        "Affected_days": int(
            (
                plot_df[column] > 0
            ).sum()
        ),
    })


qf_summary_df = (
    pd.DataFrame(qf_summary_rows)
    .sort_values(
        "Maximum_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)


if SAVE_OUTPUTS:
    qf_summary_df.to_csv(
        OUTPUT_TABLE_DIR
        / "vnp46a1_qf_dnb_flag_summary.csv",
        index=False,
    )


display(
    qf_summary_df.round(4)
)


fig_qf = go.Figure()

fig_qf.add_trace(
    go.Bar(
        x=qf_summary_df["Flag"],
        y=qf_summary_df["Mean_pct"],
        name="Daily mean",
        marker_color="#1f77b4",
    )
)

fig_qf.add_trace(
    go.Bar(
        x=qf_summary_df["Flag"],
        y=qf_summary_df["Maximum_pct"],
        name="Maximum day",
        marker_color="#d62728",
    )
)

fig_qf.update_layout(
    barmode="group",
    title=(
        "Spatial Incidence of VNP46A1 DNB Quality Flags"
        "<br><sup>"
        "Percentages use the fixed land-and-coast denominator"
        "</sup>"
    ),
    xaxis_title="DNB quality flag",
    yaxis_title="Affected land pixels (%)",
)

fig_qf.update_xaxes(
    tickangle=-25
)

apply_figure_style(
    fig_qf,
    width=1300,
    height=650,
    top=115,
)

fig_qf.show()

export_figure(
    fig_qf,
    "fig_06_vnp46a1_qf_dnb_flags",
)

,Flag,Mean_pct,Median_pct,Maximum_pct,Affected_days
0,Substitute calibration,0.0,0.0,0.0,0
1,Out of range,0.0,0.0,0.0,0
2,Saturation,0.0,0.0,0.0,0
3,Temperature not nominal,0.0,0.0,0.0,0
4,Stray light,0.0,0.0,0.0,0
5,Bowtie deleted,0.0,0.0,0.0,0
6,Missing EV,0.0,0.0,0.0,0
7,Calibration failure,0.0,0.0,0.0,0
8,Dead detector,0.0,0.0,0.0,0


In [42]:
def safe_spearman(
    data,
    x_column,
    y_column,
):
    subset = data[
        [
            x_column,
            y_column,
        ]
    ].dropna()

    if len(subset) < 3:
        return np.nan

    return subset.corr(
        method="spearman"
    ).iloc[0, 1]


lunar_radiance_rho = safe_spearman(
    plot_df,
    "Moon_illumination_pct",
    "Radiance_strict_median",
)

vza_radiance_rho = safe_spearman(
    plot_df,
    "Sensor_zenith_abs_median",
    "Radiance_strict_median",
)

vza_observability_rho = safe_spearman(
    plot_df,
    "Sensor_zenith_abs_median",
    "Strict_observable_pct",
)

utc_observability_rho = safe_spearman(
    plot_df,
    "UTC_time_median",
    "Strict_observable_pct",
)


maximum_out_of_range = (
    band_profile_summary[
        "Maximum_out_of_range_pct"
    ].max()
)


diagnostic_summary_df = pd.DataFrame([
    {
        "Diagnostic": "Calendar completeness",
        "Value": (
            f"{len(complete_dates)} of "
            f"{len(expected_dates)} days"
        ),
        "Status": (
            "Pass"
            if len(complete_dates)
            == len(expected_dates)
            else "Review"
        ),
        "Interpretation": (
            "Missing band sets create unobservable "
            "calendar days before quality screening."
        ),
    },
    {
        "Diagnostic": "Physical-value ranges",
        "Value": (
            f"Maximum out-of-range pixels: "
            f"{maximum_out_of_range:.4f}%"
        ),
        "Status": (
            "Pass"
            if maximum_out_of_range <= 0.01
            else "Review scaling"
        ),
        "Interpretation": (
            "Large violations usually indicate an "
            "incorrect scale factor or unmasked fill values."
        ),
    },
    {
        "Diagnostic": "Median strict observability",
        "Value": (
            f"{plot_df['Strict_observable_pct'].median():.2f}%"
        ),
        "Status": "Descriptive",
        "Interpretation": (
            "This is the median land area directly supported "
            "by the strict VNP46A1 observation screen."
        ),
    },
    {
        "Diagnostic": "Low-observability days",
        "Value": (
            f"{int((plot_df['Strict_observable_pct'] < 10).sum())} "
            "days below 10%"
        ),
        "Status": "Review",
        "Interpretation": (
            "These dates cannot support spatially representative "
            "radiance interpretation."
        ),
    },
    {
        "Diagnostic": "Cloud-dominated days",
        "Value": (
            f"{int((plot_df['Cloudy_pct'] > 50).sum())} "
            "days above 50% cloudy"
        ),
        "Status": "Descriptive",
        "Interpretation": (
            "Cloud-driven missingness is part of the evidence "
            "and should not be interpolated silently."
        ),
    },
    {
        "Diagnostic": "Moon–radiance association",
        "Value": (
            f"Spearman ρ = {lunar_radiance_rho:.3f}"
        ),
        "Status": (
            "Review"
            if (
                np.isfinite(
                    lunar_radiance_rho
                )
                and abs(
                    lunar_radiance_rho
                ) >= 0.30
            )
            else "Limited association"
        ),
        "Interpretation": (
            "Association indicates residual lunar influence "
            "in the at-sensor radiance."
        ),
    },
    {
        "Diagnostic": "VZA–radiance association",
        "Value": (
            f"Spearman ρ = {vza_radiance_rho:.3f}"
        ),
        "Status": (
            "Review"
            if (
                np.isfinite(
                    vza_radiance_rho
                )
                and abs(
                    vza_radiance_rho
                ) >= 0.30
            )
            else "Limited association"
        ),
        "Interpretation": (
            "Association is consistent with angular sensitivity "
            "and possible anisotropy."
        ),
    },
    {
        "Diagnostic": "VZA–observability association",
        "Value": (
            f"Spearman ρ = {vza_observability_rho:.3f}"
        ),
        "Status": "Descriptive",
        "Interpretation": (
            "This tests whether usable spatial coverage changes "
            "systematically with viewing geometry."
        ),
    },
    {
        "Diagnostic": "UTC–observability association",
        "Value": (
            f"Spearman ρ = {utc_observability_rho:.3f}"
        ),
        "Status": "Descriptive",
        "Interpretation": (
            "Acquisition-time variability may coincide with "
            "different granules or viewing configurations."
        ),
    },
    {
        "Diagnostic": "Land/water classification stability",
        "Value": (
            f"Maximum daily change: "
            f"{plot_df['Land_water_change_pct'].max():.3f}%"
        ),
        "Status": (
            "Pass"
            if (
                plot_df[
                    "Land_water_change_pct"
                ].max() <= 1
            )
            else "Review"
        ),
        "Interpretation": (
            "Large changes indicate inconsistent support masks "
            "or changing quality-field availability."
        ),
    },
])


critical_days_df = (
    plot_df[
        [
            "Date",
            "Strict_observable_pct",
            "Inclusive_observable_pct",
            "Cloudy_pct",
            "QF_DNB_good_pct",
            "Sensor_zenith_abs_median",
            "Moon_illumination_pct",
            "Radiance_strict_median",
        ]
    ]
    .sort_values(
        "Strict_observable_pct"
    )
    .head(15)
    .reset_index(drop=True)
)


if SAVE_OUTPUTS:
    diagnostic_summary_df.to_csv(
        OUTPUT_TABLE_DIR
        / "vnp46a1_diagnostic_summary.csv",
        index=False,
    )

    critical_days_df.to_csv(
        OUTPUT_TABLE_DIR
        / "vnp46a1_low_observability_days.csv",
        index=False,
    )


display(diagnostic_summary_df)

display(
    critical_days_df.round(3)
)

,Diagnostic,Value,Status,Interpretation
0,Calendar completeness,180 of 180 days,Pass,Missing band sets create unobservable calendar...
1,Physical-value ranges,Maximum out-of-range pixels: 0.0000%,Pass,Large violations usually indicate an incorrect...
2,Median strict observability,2.89%,Descriptive,This is the median land area directly supporte...
3,Low-observability days,107 days below 10%,Review,These dates cannot support spatially represent...
4,Cloud-dominated days,138 days above 50% cloudy,Descriptive,Cloud-driven missingness is part of the eviden...
5,Moon–radiance association,Spearman ρ = 0.803,Review,Association indicates residual lunar influence...
6,VZA–radiance association,Spearman ρ = 0.160,Limited association,Association is consistent with angular sensiti...
7,VZA–observability association,Spearman ρ = -0.049,Descriptive,This tests whether usable spatial coverage cha...
8,UTC–observability association,Spearman ρ = 0.058,Descriptive,Acquisition-time variability may coincide with...
9,Land/water classification stability,Maximum daily change: 4.575%,Review,Large changes indicate inconsistent support ma...


,Date,Strict_observable_pct,Inclusive_observable_pct,Cloudy_pct,QF_DNB_good_pct,Sensor_zenith_abs_median,Moon_illumination_pct,Radiance_strict_median
0,2013-11-07,0.0,0.0,100.000,100.0,0.638,0.225,NaN
1,2013-08-24,0.0,0.0,100.000,100.0,0.646,0.832,NaN
2,2013-08-07,0.0,0.0,100.000,100.0,0.484,0.009,NaN
3,2013-08-03,0.0,0.0,100.000,100.0,0.637,0.088,NaN
4,2013-08-23,0.0,0.0,100.000,100.0,0.483,0.905,NaN
5,2013-07-19,0.0,0.0,100.000,100.0,0.469,0.862,NaN
6,2013-06-16,0.0,0.0,100.000,100.0,0.639,0.498,NaN
7,2013-06-17,0.0,0.0,100.000,100.0,0.474,0.600,NaN
8,2013-06-19,0.0,0.0,100.000,100.0,0.197,0.799,NaN
9,2013-08-22,0.0,0.0,100.000,100.0,0.198,0.963,NaN
